<a href="https://colab.research.google.com/github/0szgn0/Dataleague-show-noshow-ML-alghoritm/blob/main/Dataleague.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

data_league26_path = kagglehub.competition_download('data-league26')

print('Data source import complete.')


In [ ]:
import kagglehub
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
import optuna
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
print("szgn")

In [ ]:

path = kagglehub.competition_download("data-league26")
train_df = pd.read_csv(f"{path}/appointments_train.csv")
test_df = pd.read_csv(f"{path}/appointments_test.csv")
patients_df = pd.read_csv(f"{path}/patients.csv")
clinics_df = pd.read_csv(f"{path}/clinics.csv")
print("szgn")

In [ ]:
train_merged = train_df.merge(patients_df, on='patient_id', how='left')
train_merged = train_merged.merge(clinics_df, on='clinic_id', how='left')
test_merged = test_df.merge(patients_df, on='patient_id', how='left')
test_merged = test_merged.merge(clinics_df, on='clinic_id', how='left')
print("szgn")

In [ ]:
display(train_merged. head())

In [ ]:
print("merged train shape:", train_merged.shape)
print("merged test shape:", test_merged.shape)

In [ ]:
# Check data types and general structure
print("Data Types and General Info:")
train_merged.info()
print("-" * 30)

# Check the count of missing values (NaNs)
print("Missing Values (NaN) Counts:")
missing_values = train_merged.isnull().sum()
print(missing_values[missing_values > 0]) # Shows only columns with missing values

In [ ]:
# Fix columns
for df in [train_merged, test_merged]:
    df.drop(columns=['specialty_y'], inplace=True)
    df.rename(columns={'specialty_x': 'specialty'}, inplace=True)

# Fill NaNs
train_merged['sms_lead_hours'] = train_merged['sms_lead_hours'].fillna(0)
test_merged['sms_lead_hours'] = test_merged['sms_lead_hours'].fillna(0)

In [ ]:
train_merged = train_merged.drop(columns=['specialty_y'], errors='ignore')
test_merged = test_merged.drop(columns=['specialty_y'], errors='ignore')

train_merged = train_merged.rename(columns={'specialty_x': 'specialty'}, errors='ignore')
test_merged = test_merged.rename(columns={'specialty_x': 'specialty'}, errors='ignore')

In [ ]:
train_merged.info()

In [ ]:
# String to Datetime
for col in ['appointment_datetime', 'booking_datetime']:
    train_merged[col] = pd.to_datetime(train_merged[col])
    test_merged[col] = pd.to_datetime(test_merged[col])

# Feature Engineering
for df in [train_merged, test_merged]:
    df['lead_days'] = (df['appointment_datetime'] - df['booking_datetime']).dt.days
    df['apt_month'] = df['appointment_datetime'].dt.month
    df['apt_dayofweek'] = df['appointment_datetime'].dt.dayofweek

# Drop raw dates
drop_dates = ['appointment_datetime', 'booking_datetime']
train_merged.drop(columns=drop_dates, inplace=True)
test_merged.drop(columns=drop_dates, inplace=True)

In [ ]:
# Encoding
cat_cols = ['specialty', 'booking_channel', 'appointment_type', 'sex']

train_merged = pd.get_dummies(train_merged, columns=cat_cols, drop_first=True)
test_merged = pd.get_dummies(test_merged, columns=cat_cols, drop_first=True)

# Alignment
train_merged, test_merged = train_merged.align(test_merged, join='left', axis=1, fill_value=0)
test_merged.drop(columns=['label_noshow'], inplace=True)

# Drop IDs & Noisy Features
train_merged.drop(columns=['appointment_id', 'patient_id', 'area_id'], inplace=True)
test_merged.drop(columns=['patient_id', 'area_id'], inplace=True)

In [ ]:
train_merged.info()

In [ ]:
X = train_merged.drop(columns=['label_noshow'])
y = train_merged['label_noshow']
X_test = test_merged.drop(columns=['appointment_id'])

# Validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

# Model params
params = {
    'objective': 'binary',
    'is_unbalance': True,
    'learning_rate': 0.05,
    'n_estimators': 500,
    'random_state': 42
}

# Training loop
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)])

    val_preds = model.predict_proba(X_va)[:, 1]
    oof_preds[val_idx] = val_preds

    test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits

    fold_score = average_precision_score(y_va, val_preds)
    print(f"Fold {fold+1}: {fold_score:.4f}")

# Final Score
final_score = average_precision_score(y, oof_preds)
print(f"Overall PR-AUC: {final_score:.4f}")

In [ ]:
feature_imp = pd.DataFrame(sorted(zip(model.feature_importances_, X.columns)), columns=['Value','Feature'])

plt.figure(figsize=(10, 8))
sns.barplot(x="Value", y="Feature", data=feature_imp.sort_values(by="Value", ascending=False).head(20))
plt.title('LightGBM - Top 20 Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
for df in [train_merged, test_merged]:
    df['risk_distance_lead'] = df['distance_km'] * df['lead_time_hours']

    df['clinic_stress'] = df['clinic_load_ratio'] * df['wait_mins_est']

    df['age_ses_ratio'] = df['age'] / (df['ses_score'] + 0.1)

X = train_merged.drop(columns=['label_noshow'])
X_test = test_merged.drop(columns=['appointment_id'])

print("new column count:", X.shape[1])

In [ ]:
# Reset arrays for new size
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

# CV setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Model params
params = {
    'objective': 'binary',
    'is_unbalance': True,
    'learning_rate': 0.05,
    'n_estimators': 500,
    'random_state': 42
}

# Training loop
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)])

    val_preds = model.predict_proba(X_va)[:, 1]
    oof_preds[val_idx] = val_preds

    test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits

    fold_score = average_precision_score(y_va, val_preds)
    print(f"Fold {fold+1}: {fold_score:.4f}")

# Final score
final_score = average_precision_score(y, oof_preds)
print(f"New Overall PR-AUC: {final_score:.4f}")

In [ ]:
for df in [train_merged, test_merged]:
    # 1. Risk Momentum
    df['risk_momentum'] = df['prior_noshow_count'] / (df['prior_appt_count'] + 1)

    # 0: SMS couldnt reach, 1: last day, 2: 1-3 days ago, 3: way before
    conditions = [
        (df['sms_lead_hours'] == 0),
        (df['sms_lead_hours'] > 0) & (df['sms_lead_hours'] <= 24),
        (df['sms_lead_hours'] > 24) & (df['sms_lead_hours'] <= 72),
        (df['sms_lead_hours'] > 72)
    ]
    choices = ['no_sms', 'last_day_sms', 'few_days_sms', 'early_sms']

    df['sms_profile'] = np.select(conditions, choices, default='unknown')

# sms_profile encoding
train_merged = pd.get_dummies(train_merged, columns=['sms_profile'], drop_first=True)
test_merged = pd.get_dummies(test_merged, columns=['sms_profile'], drop_first=True)

# Alignment
train_merged, test_merged = train_merged.align(test_merged, join='left', axis=1, fill_value=0)
if 'label_noshow' in test_merged.columns:
    test_merged.drop(columns=['label_noshow'], inplace=True)

print(f"New column count: {train_merged.shape[1]}")

In [ ]:
# Check date
print(f"Train max date: {train_df['appointment_datetime'].max()}")
print(f"Test min date: {test_df['appointment_datetime'].min()}")

In [ ]:
train_merged.columns

In [ ]:
X = train_merged.drop(columns=['label_noshow'], errors='ignore')
y = train_merged['label_noshow']

X_test = test_merged[X.columns]

tss = TimeSeriesSplit(n_splits=5)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

print(f"column count: {X.shape[1]}")

for fold, (train_idx, val_idx) in enumerate(tss.split(X)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.log_evaluation(0)])

    val_preds = model.predict_proba(X_va)[:, 1]
    oof_preds[val_idx] = val_preds
    test_preds += model.predict_proba(X_test)[:, 1] / 5

    fold_score = average_precision_score(y_va, val_preds)
    print(f"Fold {fold+1}: {fold_score:.4f}")

# Final Score
valid_idx = np.where(oof_preds > 0)[0]
final_score = average_precision_score(y.iloc[valid_idx], oof_preds[valid_idx])
print(f"\nTime Based PR-AUC: {final_score:.4f}")

In [ ]:
def objective(trial):
    # Parametre serch
    opt_params = {
        'objective': 'binary',
        'metric': 'average_precision',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': 42,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 5.0) # Sınıf dengesizliği için kritik
    }

    tss_opt = TimeSeriesSplit(n_splits=3)
    scores = []

    for train_idx, val_idx in tss_opt.split(X):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(**opt_params, n_estimators=500)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])

        preds = model.predict_proba(X_va)[:, 1]
        scores.append(average_precision_score(y_va, preds))

    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15)

print("\nBest Score:", study.best_value)
print("Best Parametres", study.best_params)

In [ ]:
global_mean = train_merged['label_noshow'].mean()

agg = train_merged.groupby('clinic_id')['label_noshow'].agg(['count', 'mean'])
counts = agg['count']
means = agg['mean']

weight = 10
smoothed_risk = (counts * means + weight * global_mean) / (counts + weight)

train_merged['clinic_risk_score'] = train_merged['clinic_id'].map(smoothed_risk).fillna(global_mean)
test_merged['clinic_risk_score'] = test_merged['clinic_id'].map(smoothed_risk).fillna(global_mean)

train_merged.drop(columns=['clinic_id'], inplace=True, errors='ignore')
test_merged.drop(columns=['clinic_id'], inplace=True, errors='ignore')

In [ ]:
best_params = {
    'objective': 'binary',
    'metric': 'average_precision',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'random_state': 42,
    'learning_rate': 0.016620272513595675,
    'num_leaves': 20,
    'feature_fraction': 0.9889451475970685,
    'bagging_fraction': 0.8197562449308788,
    'min_child_samples': 29,
    'scale_pos_weight': 2.040125663579343,
    'n_estimators': 600
}

lgbm_model = lgb.LGBMClassifier(**best_params)
lgbm_model.fit(X, y)

final_test_preds = lgbm_model.predict_proba(X_test)[:, 1]

In [ ]:
cb_oof_preds = np.zeros(len(X))
cb_test_preds = np.zeros(len(X_test))

# CatBoost Parametres
cb_params = {
    'iterations': 1000,
    'learning_rate': 0.03,
    'eval_metric': 'PRAUC',
    'scale_pos_weight': 2.04,
    'random_seed': 42,
    'verbose': 0
}

# TimeSeriesSplit
for fold, (train_idx, val_idx) in enumerate(tss.split(X)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    model_cb = CatBoostClassifier(**cb_params)
    model_cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=50)

    val_preds = model_cb.predict_proba(X_va)[:, 1]
    cb_oof_preds[val_idx] = val_preds

    cb_test_preds += model_cb.predict_proba(X_test)[:, 1] / 5

    fold_score = average_precision_score(y_va, val_preds)
    print(f"CatBoost Fold {fold+1}: {fold_score:.4f}")

# CatBoost score
valid_idx = np.where(cb_oof_preds > 0)[0]
cb_final_score = average_precision_score(y.iloc[valid_idx], cb_oof_preds[valid_idx])
print(f"\nCatBoost PR-AUC: {cb_final_score:.4f}")

ensemble_test_preds = (final_test_preds * 0.5) + (cb_test_preds * 0.5)

submission_ensemble = pd.DataFrame({
    'appointment_id': test_df['appointment_id'],
    'label_noshow': ensemble_test_preds
})

submission_ensemble.to_csv('submission.csv', index=False)

In [ ]:
X_fi = train_merged.drop(columns=['label_noshow'], errors='ignore')
y_fi = train_merged['label_noshow']

model_fi = lgb.LGBMClassifier(random_state=42, n_estimators=150, verbose=-1)
model_fi.fit(X_fi, y_fi)

feature_imp = pd.DataFrame({
    'Feature': X_fi.columns,
    'importance lvl': model_fi.feature_importances_
}).sort_values(by='importance lvl', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x="importance lvl", y="Feature", data=feature_imp.head(25), palette="viridis")
plt.title("LightGBM (Feature Importance)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
train_merged['age_lead_risk'] = train_merged['age'] * train_merged['lead_time_hours']
test_merged['age_lead_risk'] = test_merged['age'] * test_merged['lead_time_hours']

train_merged['transport_hardship'] = train_merged['distance_km'] / (train_merged['ses_score'] + 0.01)
test_merged['transport_hardship'] = test_merged['distance_km'] / (test_merged['ses_score'] + 0.01)

train_merged['habitual_wait_risk'] = train_merged['prior_noshow_rate'] * train_merged['lead_time_hours']
test_merged['habitual_wait_risk'] = test_merged['prior_noshow_rate'] * test_merged['lead_time_hours']

print("3 new coulmns")

In [ ]:
train_merged.info()

In [ ]:
xgb_params = {
    'objective': 'binary:logistic', 'eval_metric': 'aucpr', 'random_state': 42,
    'learning_rate': 0.02, 'max_depth': 6, 'scale_pos_weight': 2.04, 'n_estimators': 600
}
xgb_oof = np.zeros(len(X))
xgb_test_preds = np.zeros(len(X_test))

for train_idx, val_idx in tss.split(X):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    model_xgb = XGBClassifier(**xgb_params)
    model_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

    xgb_oof[val_idx] = model_xgb.predict_proba(X_va)[:, 1]
    xgb_test_preds += model_xgb.predict_proba(X_test)[:, 1] / 5

valid_idx = np.where(xgb_oof > 0)[0]

xgb_score = average_precision_score(y.iloc[valid_idx], xgb_oof[valid_idx])

print(f"XGBoost PR-AUC: {xgb_score:.5f}")